In [1]:
# 라이브러리 임포트

import platform
import json
import os
from pprint import pprint
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud
from itertools import combinations
from collections import Counter
import networkx as nx


# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False


In [3]:
import pandas as pd
import ast

# 1. CSV 불러오기
portfolio = pd.read_csv("../data/portfolio.csv")
profile = pd.read_csv("../data/profile.csv")
transcript = pd.read_csv("../data/transcript.csv")

# 2. transcript의 value 컬럼 파싱
def parse_value(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return {}

parsed_value = transcript["value"].apply(parse_value)

# 3. value 안의 주요 값 추출
transcript["offer_id"] = parsed_value.apply(
    lambda d: d.get("offer id") if "offer id" in d else d.get("offer_id")
)
transcript["amount"] = parsed_value.apply(lambda d: d.get("amount"))
transcript["reward_value"] = parsed_value.apply(lambda d: d.get("reward"))

# 4. profile / portfolio와 컬럼명 충돌 방지 위해 rename
profile_renamed = profile.rename(columns={"id": "customer_id"})
portfolio_renamed = portfolio.rename(columns={"id": "offer_id", "reward": "offer_reward"})

# 5. transcript의 person도 customer_id로 맞추기
transcript_renamed = transcript.rename(columns={"person": "customer_id"})

# 6. 고객 정보 merge
merged = transcript_renamed.merge(
    profile_renamed,
    on="customer_id",
    how="left"
)

# 7. 오퍼 정보 merge
merged = merged.merge(
    portfolio_renamed,
    on="offer_id",
    how="left"
)

# 8. 날짜형 변환 (선택)
merged["became_member_on"] = pd.to_datetime(
    merged["became_member_on"],
    format="%Y%m%d",
    errors="coerce"
)

# 9. 결과 저장
merged.to_csv("starbucks_merged.csv", index=False, encoding="utf-8-sig")

print("통합 완료:", merged.shape)
print(merged.head())

통합 완료: (306534, 19)
   Unnamed: 0_x                       customer_id           event  \
0             0  78afa995795e4d85b5d9ceeca43f5fef  offer received   
1             1  a03223e636434f42ac4c3df47e8bac43  offer received   
2             2  e2127556f4f64592b11af22de27a7932  offer received   
3             3  8ec6ce2a7e7949b1bf142def7d0e0586  offer received   
4             4  68617ca6246f4fbc85e91a2a49552598  offer received   

                                              value  time  \
0  {'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}     0   
1  {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}     0   
2  {'offer id': '2906b810c7d4411798c6938adc9daaa5'}     0   
3  {'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}     0   
4  {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}     0   

                           offer_id  amount  reward_value  Unnamed: 0_y  \
0  9b98b8c7a33c4b65b9aebfe6a799e6d9     NaN           NaN             3   
1  0b1e1539f2cc45b7b9fa7c272da2e1d7     NaN     

In [4]:
df = pd.read_csv("starbucks_merged.csv")

print(df.columns.tolist())
print(df.dtypes)
print(df.head())

['Unnamed: 0_x', 'customer_id', 'event', 'value', 'time', 'offer_id', 'amount', 'reward_value', 'Unnamed: 0_y', 'gender', 'age', 'became_member_on', 'income', 'Unnamed: 0', 'offer_reward', 'channels', 'difficulty', 'duration', 'offer_type']
Unnamed: 0_x          int64
customer_id             str
event                   str
value                   str
time                  int64
offer_id                str
amount              float64
reward_value        float64
Unnamed: 0_y          int64
gender                  str
age                   int64
became_member_on        str
income              float64
Unnamed: 0          float64
offer_reward        float64
channels                str
difficulty          float64
duration            float64
offer_type              str
dtype: object
   Unnamed: 0_x                       customer_id           event  \
0             0  78afa995795e4d85b5d9ceeca43f5fef  offer received   
1             1  a03223e636434f42ac4c3df47e8bac43  offer received   
2     

In [6]:
print(df.shape)
print(df.isna().sum())

(306534, 19)
Unnamed: 0_x             0
customer_id              0
event                    0
value                    0
time                     0
offer_id            138953
amount              167581
reward_value        272955
Unnamed: 0_y             0
gender               33772
age                      0
became_member_on         0
income               33772
Unnamed: 0          138953
offer_reward        138953
channels            138953
difficulty          138953
duration            138953
offer_type          138953
dtype: int64
